# 04. Low-Rank Decomposition: SVD, ASVD, Tucker, Monarch, and Tensor Train

- **Category:** 15_compression_peft_and_speculative_inference / 01_pruning_sparsity_lowrank_and_distillation
- **Lineage & Citations:** Eckart & Young (1936, *The approximation of one matrix by another of lower rank*, Psychometrika); Yuan et al. (2023, *ASVD: Activation-aware Singular Value Decomposition for Compressing Large Language Models*, arXiv:2310.13537); Dao et al. (2022, *Monarch: Expressive Structured Matrices for Sub-Quadratic Neural Networks*, ICML).
- **Purpose:** Formulate low-rank matrix and tensor factorizations for linear layer compression: classical Truncated SVD, activation-scaled ASVD for outlier preservation, and hardware-efficient block-permuted Monarch factorizations.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [03. LLM One-Shot Pruning: Wanda and SparseGPT](03_llm_oneshot_pruning_wanda_and_sparsegpt.ipynb) $\to` `🔗 Downstream Syntheses:` [05. Knowledge Distillation: Logit, Feature, and MiniLLM](05_knowledge_distillation_logit_feature_and_minillm.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT apply low-rank factorization when target rank $r > \frac{d_{\text{in}} d_{\text{out}}}{d_{\text{in}} + d_{\text{out}}}$ (factored representation exceeds original parameter count and doubles memory bandwidth); do NOT use vanilla SVD on LLM linear projections without activation variance scaling (outlier degradation destroys cross-entropy loss).

## Mathematical Foundations & Governing Equations

1. **Truncated SVD (Eckart-Young Optimal Approximation)**:
   $$W = U \Sigma V^\top \implies W_r = U_r \Sigma_r V_r^\top = \sum_{i=1}^r \sigma_i u_i v_i^\top, \quad \|W - W_r\|_F^2 = \sum_{i=r+1}^{\min(m,n)} \sigma_i^2$$
2. **Activation-Aware SVD (ASVD)**:
   $$W' = W \cdot \text{diag}(S), \quad W' \approx U_r \Sigma_r V_r^\top, \quad W_{\text{asvd}} = (U_r \Sigma_r V_r^\top) \cdot \text{diag}(S)^{-1}$$
3. **Monarch Factorization**:
   $$M = P_1 B_1 P_2 B_2, \quad \text{Params} = 2 N \sqrt{N} \ll N^2$$

### Asymptotic Complexity
- SVD Decomposition: $\mathcal{O}(m n \min(m, n))$ one-time offline compute.
- Factored Inference: $\mathcal{O}(B(m + n)r)$ vs. $\mathcal{O}(B m n)$ dense FLOPs.

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} golden tensors from {WEIGHTS_PATH.name}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def truncated_svd(w: torch.Tensor, rank: int = 4):
    U, S, Vh = torch.linalg.svd(w, full_matrices=False)
    U_r, S_r, Vh_r = U[:, :rank], S[:rank], Vh[:rank, :]
    w_recon = (U_r * S_r.unsqueeze(0)) @ Vh_r
    frob_err = torch.norm(w - w_recon, p="fro")
    return w_recon, S_r, frob_err

def asvd_decompose(w: torch.Tensor, act_scales: torch.Tensor, rank: int = 4):
    s = torch.clamp(act_scales, min=1e-5)
    w_scaled = w * s.unsqueeze(0)
    U, S, Vh = torch.linalg.svd(w_scaled, full_matrices=False)
    U_r, S_r, Vh_r = U[:, :rank], S[:rank], Vh[:rank, :]
    w_scaled_r = (U_r * S_r.unsqueeze(0)) @ Vh_r
    w_asvd = w_scaled_r / s.unsqueeze(0)
    return w_asvd, S_r

def monarch_forward(x: torch.Tensor, b1: torch.Tensor, b2: torch.Tensor) -> torch.Tensor:
    batch = x.shape[0]
    x_blk = x.view(batch, 4, 4)
    h1 = torch.einsum("bki,kij->bkj", x_blk, b1)
    h1_perm = h1.permute(0, 2, 1).contiguous()
    h2 = torch.einsum("bki,kij->bkj", h1_perm, b2)
    return h2.reshape(batch, 16)

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

# 2. Truncated SVD parity
w_svd, s_svd, frob_svd = truncated_svd(weights["w"], rank=4)
assert float(torch.max(torch.abs(w_svd - weights["w_svd"]))) < 1e-5
assert float(torch.max(torch.abs(s_svd - weights["s_svd"]))) < 1e-5

# 3. ASVD parity
w_asvd, s_asvd = asvd_decompose(weights["w"], weights["act_scales"], rank=4)
assert float(torch.max(torch.abs(w_asvd - weights["w_asvd"]))) < 1e-5
assert float(torch.max(torch.abs(s_asvd - weights["s_asvd"]))) < 1e-5

# 4. Monarch forward parity
y_monarch = monarch_forward(weights["x"], weights["b1"], weights["b2"])
assert float(torch.max(torch.abs(y_monarch - weights["y_monarch"]))) < 1e-5
print("All Low-Rank SVD, ASVD, and Monarch implementations verified against seed-42 golden tensors!")

In [ ]:
# Empirical evaluation: SVD vs ASVD Output Approximation Error
x = weights["x"]
y_orig = x @ weights["w"].T
err_svd = float(torch.norm(y_orig - x @ w_svd.T)**2 / (torch.norm(y_orig)**2 + 1e-8))
err_asvd = float(torch.norm(y_orig - x @ w_asvd.T)**2 / (torch.norm(y_orig)**2 + 1e-8))
print(f"[1] Truncated SVD Rank-4 Relative Output Error: {err_svd:.4f}")
print(f"[2] ASVD Rank-4 Relative Output Error:          {err_asvd:.4f}")
assert err_svd > 0 and err_asvd > 0, "Errors must be strictly positive"
param_dense = weights["w"].numel()
param_factored = (16 + 16) * 4
print(f"Compression: {param_dense} dense params -> {param_factored} factored params ({param_factored/param_dense*100:.1f}%)")

In [ ]:
# Pathological Misuse: Setting Rank Above Critical Breakeven Threshold
# When rank r > (m * n) / (m + n), factored GEMM increases both parameter count and FLOPs!
m, n = 16, 16
r_breakeven = (m * n) / (m + n)
r_bad = 12
params_dense = m * n
params_factored = (m + n) * r_bad
print(f"Breakeven rank threshold: {r_breakeven:.1f}")
print(f"Rank {r_bad} factored parameters: {params_factored} vs Dense: {params_dense} (+{((params_factored-params_dense)/params_dense)*100:.1f}% overhead!)")
assert params_factored > params_dense, "Factoring at high rank must yield parameter bloat!"
print("Pathology demonstrated: Low-rank factorization requires r < (m*n)/(m+n) to avoid bloating compute and memory.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Eckart-Young Optimality**: Truncated SVD provides the mathematically optimal low-rank Frobenius approximation, decomposing a dense matrix into two sequential GEMM operations.
2. **Activation Awareness**: ASVD weights feature columns by input activation variance before factorization, preventing severe output degradation caused by outlier feature channels.
3. **Monarch Structure**: Two block-diagonal stages with permutation achieve $\mathcal{O}(N \sqrt{N})$ computation with hardware-friendly batched GEMMs.

---
### Cross-Domain Generalisation Matrix
- **PEFT**: LoRA is a low-rank adapter parameterization $\Delta W = B A$ initialized with Gaussian $A$ and zero $B$.
- **Quantum & Chemistry**: Tensor Train (TT) is isomorphic to Matrix Product States (MPS) in quantum many-body physics.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.md](../../../markdown/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.md)
- **PyTorch 2.x Module:** [04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.py](../../../pytorch2/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.py)
- **Functional JAX Module:** [04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.py](../../../jax/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.py)
- **Native Rust Candle Track:** [04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.rs](../../../candle/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.rs)
- **Hardware Kernel (CPU / WGSL):** [04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.rs](../../../kernels/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.rs)
- **Golden SafeTensors:** [04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.safetensors](../../../weights/15_compression_peft_and_speculative_inference/01_pruning_sparsity_lowrank_and_distillation/04_low_rank_decomposition_svd_asvd_tucker_monarch_and_tt.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. LLM Oneshot Pruning Wanda and Sparsegpt](03_llm_oneshot_pruning_wanda_and_sparsegpt.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 05. Knowledge Distillation Logit Feature and Minillm](05_knowledge_distillation_logit_feature_and_minillm.ipynb) &rarr;